# Laboratório Prático: Prevendo o Preço de um Imóvel
## Regressão Múltipla, Variáveis Dummy, Limpeza e Seaborn

**Disciplina:** ADS308  
**Prof.:** João Vítor Rodrigues de Vasconcelos  
**Curso:** Engenharia de Computação / Análise e Desenvolvimento de Sistemas  

---

### Objetivos
- Consolidar o uso da Regressão Linear Múltipla
- Realizar Análise Exploratória de Dados (EDA) com Seaborn
- Lidar com Variáveis Categóricas via One-Hot Encoding (Variáveis Dummy)
- Limpar dados inconsistentes e ruidosos (Data Wrangling)
- Calcular e interpretar o RMSE no contexto financeiro

## O Cenário

A imobiliária de Viçosa-MG expandiu sua base de dados. Temos acesso a **1000 registros históricos** de vendas, incluindo o tipo do imóvel.

**Features:** Area_m2, Num_Quartos, Tipo_Imovel, Idade_Imovel  
**Target:** Preco_Mil (valor em milhares de R$)

## Passo 1: Carregamento e Inspeção dos Dados

In [ ]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns

from sklearn.model_selection import train_test_split
from sklearn.linear_model import LinearRegression
from sklearn.metrics import mean_squared_error, r2_score

# Carregando o dataset
df = pd.read_csv("dataset_imoveis_vicosa.csv")
print("Dados carregados com sucesso! Tamanho:", df.shape)

In [ ]:
# Verificando os tipos de dados e valores nulos
print(df.info())

In [ ]:
# Inspecionando as primeiras linhas
df.head(10)

## Passo 2: Análise Exploratória com Seaborn

### 2.1 Distribuição dos Preços

In [ ]:
sns.set_theme(style="whitegrid")

plt.figure(figsize=(8, 5))
sns.histplot(df["Preco_Mil"], kde=True, bins=20, color="blue")
plt.title("Distribuição dos Preços dos Imóveis")
plt.xlabel("Preço (Milhares de R$)")
plt.ylabel("Frequência")
plt.show()

### 2.2 Boxplot: Preço por Tipo de Imóvel

In [ ]:
plt.figure(figsize=(10, 6))
sns.boxplot(x="Tipo_Imovel", y="Preco_Mil", data=df)
plt.title("Preço do Imóvel por Tipo (Categoria)")
plt.xticks(rotation=15)
plt.show()

**Análise Visual:** Qual tipo de imóvel possui a maior mediana de preço? Anote abaixo.

*Resposta:*


### 2.3 Scatter Plot: Área vs Preço

In [ ]:
plt.figure(figsize=(10, 6))
sns.scatterplot(x="Area_m2", y="Preco_Mil", hue="Tipo_Imovel",
                size="Num_Quartos", sizes=(20, 150), data=df, alpha=0.8)
plt.title("Relação entre Área, Tipo e Preço")
plt.legend(bbox_to_anchor=(1.05, 1), loc="upper left")
plt.tight_layout()
plt.show()

## Passo 3: Limpeza da Variável Suja (Idade_Imovel)

A coluna `Idade_Imovel` está como `object` e possui valores como "NaN", "novo", "anos" e até números negativos.

**Tarefas:**
1. Remover a string "anos" (usar `.str.replace()`)
2. Substituir "novo" por 0
3. Transformar "Desconhecida" em NaN
4. Converter a coluna para numérico e remover/corrigir idades negativas
5. Imputar os NaN com a mediana

In [ ]:
# Verificando valores únicos problemáticos
print("Tipo da coluna:", df["Idade_Imovel"].dtype)
print("\nValores únicos (amostra):")
print(df["Idade_Imovel"].unique()[:20])

In [ ]:
# TODO: Realize a limpeza da coluna Idade_Imovel

# 1. Remover a string " anos" (atenção ao espaço)
# df["Idade_Imovel"] = df["Idade_Imovel"].str.replace(" anos", "", regex=False)

# 2. Substituir "novo" por "0"
# df["Idade_Imovel"] = df["Idade_Imovel"].replace("novo", "0")

# 3. Transformar "Desconhecida" em NaN
# df["Idade_Imovel"] = df["Idade_Imovel"].replace("Desconhecida", np.nan)

# 4. Converter para numérico (errors='coerce' transforma o que não conseguir em NaN)
# df["Idade_Imovel"] = pd.to_numeric(df["Idade_Imovel"], errors="coerce")

# 5. Tratar negativos: substituir negativos por NaN
# df.loc[df["Idade_Imovel"] < 0, "Idade_Imovel"] = np.nan

# 6. Imputar NaN com a mediana
# mediana_idade = df["Idade_Imovel"].median()
# df["Idade_Imovel"] = df["Idade_Imovel"].fillna(mediana_idade)

print("Tipo após limpeza:", df["Idade_Imovel"].dtype)
print("Nulos restantes:", df["Idade_Imovel"].isnull().sum())
print("\nDescrição:")
print(df["Idade_Imovel"].describe())

## Passo 4: One-Hot Encoding (Variáveis Dummy)

Converter a coluna `Tipo_Imovel` (texto) em colunas binárias.

O parâmetro `drop_first=True` evita a armadilha da multicolinearidade.

In [ ]:
# Convertendo a coluna qualitativa em colunas quantitativas
df = pd.get_dummies(df, columns=["Tipo_Imovel"], drop_first=True)

# Vejam o resultado
df.head()

## Passo 5: Matriz de Correlação (Pós-Limpeza)

Apenas depois de limpar a idade e aplicar o `get_dummies`, podemos calcular as correlações.

In [ ]:
plt.figure(figsize=(10, 8))
# Calculando a correlação de Pearson
matriz_corr = df.corr(numeric_only=True)

# Plotando o Heatmap
sns.heatmap(matriz_corr, annot=True, cmap="coolwarm", fmt=".2f")
plt.title("Matriz de Correlação")
plt.show()

## Passo 6: Divisão em Treino e Teste

In [ ]:
# O Target é o Preco_Mil. Todo o resto é Feature.
X = df.drop("Preco_Mil", axis=1)
y = df["Preco_Mil"]

# Split de 80% treino / 20% teste
X_train, X_test, y_train, y_test = train_test_split(
    X, y, test_size=0.2, random_state=42
)

print(f"Treino: {X_train.shape[0]} amostras")
print(f"Teste:  {X_test.shape[0]} amostras")

## Passo 7: Treinamento do Modelo (OLS)

In [ ]:
# Instanciando o modelo
modelo = LinearRegression()

# Treinando (Calculando os Mínimos Quadrados)
modelo.fit(X_train, y_train)

# Inspecionando
print("Intercepto:", modelo.intercept_)
print("\nCoeficientes:")
for col, coef in zip(X_train.columns, modelo.coef_):
    print(f"  {col}: {coef:.4f}")

## Passo 8: Métricas de Erro (RMSE)

In [ ]:
# Predições
y_pred = modelo.predict(X_test)

# Calculando o MSE e RMSE
mse = mean_squared_error(y_test, y_pred)
rmse = np.sqrt(mse)

print(f"MSE: {mse:.2f}")
print(f"RMSE: {rmse:.2f}")
print(f"\nInterpretação: Em média, o modelo erra por R$ {rmse:.2f} mil (R$ {rmse*1000:.2f})")

## Passo 9: Coeficiente de Determinação (R²)

In [ ]:
# Calculando o R^2 no conjunto de teste
r2 = r2_score(y_test, y_pred)
print(f"R² Score: {r2:.4f} (ou {r2*100:.2f}%)")
print(f"\nInterpretação: {r2*100:.2f}% da variação nos preços é explicada pelas features.")

## Passo 10: Interpretação de Negócio

Analisem os coeficientes, o RMSE e o R². O modelo já pode ser colocado em produção na imobiliária? Justifiquem sua decisão técnica e de negócio.

*Sua análise aqui:*


## Desafio Prático: Prevendo Novos Valores

**Pergunta de negócio:** Qual é o preço estimado para um **Apartamento Padrão**, com **75m²**, **2 quartos** e **5 anos** de construção?

In [ ]:
# Verificando a ordem das colunas que o modelo espera
print("Colunas do X_train:")
print(X_train.columns.tolist())

In [ ]:
# TODO: Crie a matriz de características respeitando a ordem das colunas
# Lembre-se: preencha com 0 as categorias dummy que não se aplicam
# e com 1 a categoria correspondente a "Apartamento Padrão"

# Exemplo (ajuste conforme as colunas do seu X_train):
# caracteristicas_cliente = [[75, 2, 5, 1, 0, 0]]

# preco_previsto = modelo.predict(caracteristicas_cliente)
# print(f"Preço estimado: R$ {preco_previsto[0]:.2f} mil")
# print(f"Preço estimado: R$ {preco_previsto[0] * 1000:,.2f}")